# Synthetic Exchange Ledger Generator (Canonical First)

## Purpose

This notebook generates **mathematically and logically correct canonical transactions**
that represent realistic exchange activity (trades, deposits, withdrawals, transfers).

The generated canonical ledger can later be **intentionally corrupted**
to resemble real exchange data structures (e.g. Binance),
allowing robust testing of exchange parsers and tax logic.

This notebook is a **helper / modeling tool** and is not part of the production system.

In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta, date
from enum import Enum
from decimal import Decimal
from collections import defaultdict
import pandas as pd
import random
import uuid
import os
import bisect
import yfinance as yf

In [2]:
class TransactionType(str, Enum):
    TRADE = "Trade"
    DEPOSIT = "Deposit"
    WITHDRAWAL = "Withdrawal"
    TRANSFER = "Transfer"

In [3]:
@dataclass(frozen=True)
class CanonicalTransaction:
    id: str
    timestamp: datetime
    type: TransactionType

    asset_in: str | None
    amount_in: Decimal

    asset_out: str | None
    amount_out: Decimal

    fee_asset: str | None
    fee_amount: Decimal

    fiat_value: Decimal
    fiat_currency: str

In [4]:
class AccountState:
    def __init__(self):
        self.balances = defaultdict(Decimal)

    def has(self, asset: str, amount: Decimal) -> bool:
        return self.balances[asset] >= amount

    def apply(self, tx: CanonicalTransaction):
        if tx.asset_out and tx.amount_out > 0:
            if not self.has(tx.asset_out, tx.amount_out):
                raise ValueError("Insufficient balance")
            self.balances[tx.asset_out] -= tx.amount_out

        if tx.asset_in and tx.amount_in > 0:
            self.balances[tx.asset_in] += tx.amount_in

In [5]:
class DailyPriceProvider:
    """
    Loads daily prices from CSVs if available; 
    falls back to yfinance if missing.
    Uses only the 'Close' price.
    """
    def __init__(self, price_dir: str, symbols=None):
        self.prices = {}      # { "ADA/USD": { date: close } }
        self.price_dates = {} # { "ADA/USD": [sorted dates] }
        self.symbols = symbols or ["BTC", "ETH", "ADA", "SOL"]

        for crypto in self.symbols:
            symbol = f"{crypto}/USD"
            csv_file = os.path.join(price_dir, f"{crypto}USDT_D1.csv")
            
            if os.path.exists(csv_file):
                # Load from CSV
                df = pd.read_csv(csv_file)
                df["datetime"] = pd.to_datetime(df["datetime"]).dt.date
                df = df.sort_values("datetime")
                self.prices[symbol] = dict(zip(df["datetime"], df["close"]))
                self.price_dates[symbol] = list(df["datetime"])
            else:
                # Fallback: download via yfinance
                ticker = f"{crypto}-USD"
                print(f"Downloading {ticker} via yfinance...")
                df = yf.download(ticker, interval="1d")
                df = df.reset_index()
                df["Date"] = pd.to_datetime(df["Date"]).dt.date
                df = df.sort_values("Date")
                self.prices[symbol] = dict(zip(df["Date"], df["Close"]))
                self.price_dates[symbol] = list(df["Date"])

        # Compute safe overlapping date range
        self.min_date = max(d[0] for d in self.price_dates.values())
        self.max_date = min(d[-1] for d in self.price_dates.values())

    def clamp_start_date(self, days: int) -> datetime:
        """
        Ensures the simulation start date falls within a safe price window.
    
        Since each asset may have a different historical range,
        we compute a global overlapping window where *all* assets
        have price data available.
    
        The simulation start date is clamped so that:
        - It does not precede the earliest common available date
        - It stays within the requested lookback window
        """
        start = self.max_date - timedelta(days=days)
        if start < self.min_date:
            start = self.min_date
        return datetime.combine(start, datetime.min.time())

    def get_price(self, symbol: str, ts: datetime) -> Decimal:
        dates = self.price_dates[symbol]
        d = ts.date()
    
        idx = bisect.bisect_right(dates, d) - 1
        if idx < 0:
            raise KeyError(f"No price <= {d} for {symbol}")
    
        base_price = Decimal(str(self.prices[symbol][dates[idx]]))
        return base_price

In [6]:
FIAT = "USD"
CRYPTOS = ["BTC", "ETH", "ADA", "SOL"]

PERCENTAGES = [Decimal("0.2"), Decimal("0.5"), Decimal("0.8")]
DEPOSIT_AMOUNTS = [Decimal("500"), Decimal("10000"), Decimal("50000")]

ACTIONS = [
    TransactionType.TRADE,
    TransactionType.DEPOSIT,
    TransactionType.WITHDRAWAL,
    TransactionType.TRANSFER,
]

ACTION_WEIGHTS = [0.7, 0.1, 0.1, 0.1]
FEE_RATE = Decimal("0.001")  # 0.1%

In [7]:
def gen_deposit(ts):
    """
    Generates a fiat-only deposit.

    Deposits are intentionally limited to fiat (USD) to reflect
    common exchange behavior and to simplify accounting invariants.
    Crypto deposits can be added later if needed.
    """
    amount = random.choice(DEPOSIT_AMOUNTS)
    return CanonicalTransaction(
        str(uuid.uuid4()), ts, TransactionType.DEPOSIT,
        FIAT, amount,
        None, Decimal("0"),
        None, Decimal("0"),
        amount, FIAT
    )

In [8]:
def gen_withdrawal(account, ts):
    """
    Generates a fiat-only withdrawal.

    Withdrawals are restricted to fiat to avoid modeling
    blockchain-specific behavior (networks, confirmations, fees).
    """
    if account.balances[FIAT] <= 0:
        return None

    amount = account.balances[FIAT] * random.choice(PERCENTAGES)
    return CanonicalTransaction(
        str(uuid.uuid4()), ts, TransactionType.WITHDRAWAL,
        None, Decimal("0"),
        FIAT, amount,
        None, Decimal("0"),
        amount, FIAT
    )

In [9]:
def gen_transfer(account, ts):
    """
    Generates an internal crypto transfer.

    Transfers apply only to crypto assets and do not affect fiat.
    They represent movements between wallets or accounts.
    """
    assets = [a for a in CRYPTOS if account.balances[a] > 0]
    if not assets:
        return None

    asset = random.choice(assets)
    amount = account.balances[asset] * random.choice(PERCENTAGES)

    return CanonicalTransaction(
        str(uuid.uuid4()), ts, TransactionType.TRANSFER,
        None, Decimal("0"),
        asset, amount,
        None, Decimal("0"),
        Decimal("0"), FIAT
    )

In [10]:
def gen_trade(account, prices, ts):
    trade_type = random.choices(["FIAT", "CRYPTO"], weights=[0.6, 0.4])[0]

    if trade_type == "FIAT":
        crypto = random.choice(CRYPTOS)
        price = prices.get_price(f"{crypto}/USD", ts)
        # Randomly decide between BUY (fiat → crypto) and SELL (crypto → fiat)
        if random.choice([True, False]):
            if account.balances[FIAT] <= 0:
                return None
            spend = account.balances[FIAT] * random.choice(PERCENTAGES)
            gross_receive = spend / price
            fee = gross_receive * FEE_RATE
            net_receive = gross_receive - fee
            return CanonicalTransaction(
                str(uuid.uuid4()), ts, TransactionType.TRADE,
                crypto, net_receive,
                FIAT, spend,
                crypto, fee,
                spend, FIAT
            )
        if account.balances[crypto] <= 0:
            return None
        sell = account.balances[crypto] * random.choice(PERCENTAGES)
        gross_receive = sell * price
        fee = gross_receive * FEE_RATE
        net_receive = gross_receive - fee
        return CanonicalTransaction(
                str(uuid.uuid4()), ts, TransactionType.TRADE,
                FIAT, net_receive,
                crypto, sell,
                FIAT, fee,
                net_receive, FIAT
            )
    # Randomly select two distinct crypto assets for a crypto-to-crypto trade
    base, quote = random.sample(CRYPTOS, 2)
    if account.balances[quote] <= 0:
        return None

    base_price = prices.get_price(f"{base}/USD", ts)
    quote_price = prices.get_price(f"{quote}/USD", ts)

    spend = account.balances[quote] * random.choice(PERCENTAGES)
    gross_receive = spend * quote_price / base_price
    fee = gross_receive * FEE_RATE
    net_receive = gross_receive - fee

    return CanonicalTransaction(
                str(uuid.uuid4()), ts, TransactionType.TRADE,
                base, net_receive,
                quote, spend,
                base, fee,
                spend * quote_price, FIAT
            )

In [11]:
def simulate(days=30, tx_per_day=5, price_dir="prices"):
    account = AccountState()
    prices = DailyPriceProvider(price_dir)
    ledger = []

    ts = prices.clamp_start_date(days)

    seed = gen_deposit(ts)
    account.apply(seed)
    ledger.append(seed)

    for _ in range(days * tx_per_day):
        ts += timedelta(hours=random.randint(1, 6))
        action = random.choices(ACTIONS, weights=ACTION_WEIGHTS, k=1)[0]

        try:
            if action == TransactionType.DEPOSIT:
                tx = gen_deposit(ts)
            elif action == TransactionType.WITHDRAWAL:
                tx = gen_withdrawal(account, ts)
            elif action == TransactionType.TRANSFER:
                tx = gen_transfer(account, ts)
            else:
                tx = gen_trade(account, prices, ts)

            if tx:
                account.apply(tx)
                ledger.append(tx)

        except (ValueError, KeyError):
            pass
    rows = [canonical_to_dict(tx) for tx in ledger]
    ledger_df = pd.DataFrame(rows)

    return ledger, dict(account.balances), ledger_df
   #return ledger, dict(account.balances)

In [12]:
def canonical_to_dict(tx: CanonicalTransaction):
    return {
        "Id": tx.id,
        "Timestamp": tx.timestamp.isoformat(),
        "Type": tx.type.value,
        "AssetIn": tx.asset_in,
        "AmountIn": float(tx.amount_in),
        "AssetOut": tx.asset_out,
        "AmountOut": float(tx.amount_out),
        "FeeAsset": tx.fee_asset,
        "FeeAmount": float(tx.fee_amount),
        "FiatValueAtTimestamp": float(tx.fiat_value),
        "FiatCurrency": tx.fiat_currency,
    }

In [13]:
def test_invariants(ledger, account_balances):
    """
    Run basic invariants on generated ledger.
    """
    balances = defaultdict(Decimal)

    for tx in ledger:
        # Invariant: no negative amounts in canonical tx
        assert tx.amount_in >= 0, f"Negative amount_in in {tx}"
        assert tx.amount_out >= 0, f"Negative amount_out in {tx}"
        assert tx.fee_amount >= 0, f"Negative fee_amount in {tx}"

        # Invariant: deposit/withdrawal/transfer asset checks
        if tx.type == TransactionType.DEPOSIT:
            assert tx.asset_in == FIAT, f"Deposit must be fiat: {tx}"
        elif tx.type == TransactionType.WITHDRAWAL:
            assert tx.asset_out == FIAT, f"Withdrawal must be fiat: {tx}"
        elif tx.type == TransactionType.TRANSFER:
            assert tx.asset_out in CRYPTOS and tx.amount_out > 0, f"Transfer must be crypto: {tx}"

        # Update simulated balances
        if tx.asset_out and tx.amount_out > 0:
            balances[tx.asset_out] -= tx.amount_out
        if tx.asset_in and tx.amount_in > 0:
            balances[tx.asset_in] += tx.amount_in

        # Invariant: balances should never go negative
        for asset, amt in balances.items():
            assert amt >= 0, f"Negative balance for {asset} after {tx}"

    # Final account balances check
    for asset, amt in account_balances.items():
        assert balances[asset] == amt, f"Final balance mismatch for {asset}"
    print("All invariants passed.")

In [14]:
ledger, account_balances, _ = simulate(days=10, tx_per_day=5, price_dir="prices")
test_invariants(ledger, account_balances)

All invariants passed.


# Binance Corruption Function

This section converts the canonical ledger into a **realistic Binance-style CSV**. 

Rules applied:
- Deposits / Withdrawals / Transfers → one row, mapped to Binance column names
- Trades → 3 rows (Asset In, Asset Out, Fee)
  - Rarely split into 3 × n rows to simulate divided trades
  - Randomly pick operation names among common Binance labels
  - Amounts are proportionally divided and rounded

In [15]:
# -----------------------------
# Binance-style operation sets
# -----------------------------
TRADE_OPERATION_SETS = [
    ["Transaction Related", "Transaction Related", "Transaction Fee"],
    ["Binance Convert", "Binance Convert", "Transaction Fee"],
    ["Buy", "Sell", "Fee"],
    ["Transaction Buy", "Transaction Spend", "Transaction Fee"],
    ["Transaction Revenue", "Transaction Sold", "Transaction Fee"]
]

In [16]:
import pandas as pd
import random
from decimal import Decimal

def corrupt_to_binance_style(
    ledger,
    split_prob: float = 0.05,
    max_splits: int = 5,
    rounding: int = 8,
    seed: int | None = None
) -> pd.DataFrame:
    """
    Convert canonical ledger into a DataFrame resembling Binance CSV exports.

    Rules:
    - Deposit / Withdraw / Transfer → one row
    - Trade → 3 rows (asset_in, asset_out, fee)
        - Rarely split into 3*n rows
        - Randomly pick operation name set
        - Divide amounts proportionally if split
    """
    if seed is not None:
        random.seed(seed)

    rows = []

    for tx in ledger:
        # -----------------------------
        # Deposit / Withdraw / Transfer
        # -----------------------------
        if tx.type == TransactionType.DEPOSIT:
            rows.append({
                "Time": tx.timestamp,
                "Account": "Spot",
                "Operation": "Deposit",
                "Coin": tx.asset_in,
                "Change": round(float(tx.amount_in), rounding)
            })
            continue

        if tx.type == TransactionType.WITHDRAWAL:
            rows.append({
                "Time": tx.timestamp,
                "Account": "Spot",
                "Operation": "Withdraw",
                "Coin": tx.asset_out,
                "Change": round(float(-tx.amount_out), rounding)
            })
            continue

        if tx.type == TransactionType.TRANSFER:
            rows.append({
                "Time": tx.timestamp,
                "Account": "Spot",
                "Operation": "Transfer",
                "Coin": tx.asset_out,
                "Change": round(float(-tx.amount_out), rounding)
            })
            continue

        # -----------------------------
        # Trades
        # -----------------------------
        n_splits = 1
        if random.random() < split_prob:
            n_splits = random.randint(2, max_splits)

        op_set = random.choice(TRADE_OPERATION_SETS)
        op_in, op_out, op_fee = op_set

        splits_in = [tx.amount_in / n_splits for _ in range(n_splits)]
        splits_out = [tx.amount_out / n_splits for _ in range(n_splits)]
        splits_fee = [tx.fee_amount / n_splits for _ in range(n_splits)]

        for i in range(n_splits):
            # Asset In row
            rows.append({
                "Time": tx.timestamp,
                "Account": "Spot",
                "Operation": op_in,
                "Coin": tx.asset_in,
                "Change": round(float(splits_in[i]), rounding)
            })
            # Asset Out row
            rows.append({
                "Time": tx.timestamp,
                "Account": "Spot",
                "Operation": op_out,
                "Coin": tx.asset_out,
                "Change": round(float(-splits_out[i]), rounding)
            })
            # Fee row
            if splits_fee[i] > 0:
                rows.append({
                    "Time": tx.timestamp,
                    "Account": "Spot",
                    "Operation": op_fee,
                    "Coin": tx.fee_asset,
                    "Change": round(float(-splits_fee[i]), rounding)
                })

    df = pd.DataFrame(rows).sort_values("Time").reset_index(drop=True)
    return df

In [17]:
# Generate corrupted Binance-style DataFrame
ledger, account_balances, ledger_df = simulate(days=10, tx_per_day=5, price_dir="prices")
binance_df = corrupt_to_binance_style(ledger, split_prob=0.05, max_splits=4, rounding=8, seed=42)

# Show top rows
binance_df.head(10)

,Time,Account,Operation,Coin,Change
0,2024-03-03 00:00:00,Spot,Deposit,USD,500.000000
1,2024-03-03 11:00:00,Spot,Withdraw,USD,-250.000000
2,2024-03-03 23:00:00,Spot,Transaction Related,ADA,68.622063
3,2024-03-03 23:00:00,Spot,Transaction Related,USD,-50.000000
4,2024-03-03 23:00:00,Spot,Transaction Fee,ADA,-0.068691
5,2024-03-04 03:00:00,Spot,Binance Convert,ETH,0.044060
6,2024-03-04 03:00:00,Spot,Binance Convert,USD,-160.000000
7,2024-03-04 03:00:00,Spot,Transaction Fee,ETH,-0.000044
8,2024-03-04 08:00:00,Spot,Transaction Related,BTC,0.001872
9,2024-03-04 08:00:00,Spot,Transaction Related,ETH,-0.035248


In [19]:
import os

# Directory for saving simulated files
output_dir = "exchange_simulation"
os.makedirs(output_dir, exist_ok=True)

num_loops = 10
for i in range(1, num_loops + 1):
    # Generate canonical ledger
    ledger, account_balances, ledger_df = simulate(days=10, tx_per_day=5, price_dir="prices")
    
    # Generate Binance-style corrupted DataFrame
    binance_df = corrupt_to_binance_style(ledger, split_prob=0.05, max_splits=4, rounding=8, seed=i*42)
    
    # Save to CSV
    ledger_file = os.path.join(output_dir, f"{i}_simulated_ledger.csv")
    binance_file = os.path.join(output_dir, f"{i}_simulated_binance.csv")
    
    ledger_df.to_csv(ledger_file, index=False)
    binance_df.to_csv(binance_file, index=False)
    
    print(f"Saved loop {i}: ledger → {ledger_file}, binance → {binance_file}")

Saved loop 1: ledger → exchange_simulation\1_simulated_ledger.csv, binance → exchange_simulation\1_simulated_binance.csv
Saved loop 2: ledger → exchange_simulation\2_simulated_ledger.csv, binance → exchange_simulation\2_simulated_binance.csv
Saved loop 3: ledger → exchange_simulation\3_simulated_ledger.csv, binance → exchange_simulation\3_simulated_binance.csv
Saved loop 4: ledger → exchange_simulation\4_simulated_ledger.csv, binance → exchange_simulation\4_simulated_binance.csv
Saved loop 5: ledger → exchange_simulation\5_simulated_ledger.csv, binance → exchange_simulation\5_simulated_binance.csv
Saved loop 6: ledger → exchange_simulation\6_simulated_ledger.csv, binance → exchange_simulation\6_simulated_binance.csv
Saved loop 7: ledger → exchange_simulation\7_simulated_ledger.csv, binance → exchange_simulation\7_simulated_binance.csv
Saved loop 8: ledger → exchange_simulation\8_simulated_ledger.csv, binance → exchange_simulation\8_simulated_binance.csv
Saved loop 9: ledger → exchange_